# Where Does Matter Stop Existing?
### A quantified study of nuclear mass models and the neutron drip line

Run every cell **from the top, in order**. Each part uses results from the parts above it.

Plain-language explanations of every step are in the `docs/` folder of the repository.

## Phase 0 and Phase 1: build the dataset

Download the AME2020 mass table, keep only real measurements, and compute the binding energy of every nucleus.

In [ ]:
url <- "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20.txt"
if (!file.exists("mass_1.mas20.txt")) download.file(url, "mass_1.mas20.txt")
lines <- readLines("mass_1.mas20.txt")
length(lines)

In [ ]:
cat(head(lines, 40), sep = "\n")

1    a0dsskgw                                 A T O M I C   M A S S   A D J U S T M E N T
0                                                     DATE  3 Mar 2021 TIME 22:41
0        *********************                               A=   0 TO 295
         * file : mass.mas20 *
         *********************

   This is one file out of a series of 3 files published in:
       "The Ame2020 atomic mass evaluation (I)"   by W.J.Huang, M.Wang, F.G.Kondev, G.Audi and S.Naimi
           Chinese Physics C45, 030002, March 2021.
       "The Ame2020 atomic mass evaluation (II)"  by M.Wang, W.J.Huang, F.G.Kondev, G.Audi and S.Naimi
           Chinese Physics C45, 030003, March 2021.
                       for files : mass.mas20 : atomic masses
                                   rct1.mas20 : react and sep energies,  part 1
                                   rct2.mas20 : react and sep energies,  part 2
   A fourth file  is the "Rounded" version of the atomic mass table (the first file)
            

In [ ]:
lines <- readLines("mass_1.mas20.txt")

# Data rows: N, Z, A must all parse as integers in cols 5-19
is_data <- nchar(lines) > 100 &
           grepl("^\\s*\\d+\\s+\\d+\\s+\\d+\\s*$", substr(lines, 5, 19))

rows <- lines[is_data]
cat("Header lines:", sum(!is_data), "\n")   # expect 36
cat("Data rows:   ", length(rows), "\n")    # expect 3558

# Estimated values: '#' in the mass excess field or its uncertainty
est <- grepl("#", substr(rows, 29, 54), fixed = TRUE)
cat("Estimated (excluded):", sum(est), "\n")
cat("Measured (kept):     ", sum(!est), "\n")

m <- rows[!est]

num <- function(x, a, b) as.numeric(substr(x, a, b))

df <- data.frame(
  N        = num(m,   5,   9),
  Z        = num(m,  10,  14),
  A        = num(m,  15,  19),
  D        = num(m,  29,  42),   # mass excess, keV
  sigma    = num(m,  43,  54),   # its uncertainty, keV
  BA_table = num(m,  55,  67)    # AME's own B/A, for checking
)

DH <- 7288.97106
Dn <- 8071.31806

df$B  <- df$Z * DH + df$N * Dn - df$D
df$BA <- df$B / df$A

Header lines: 36 
Data rows:    3558 
Estimated (excluded): 1008 
Measured (kept):      2550 


In [ ]:
d <- df$BA - df$BA_table
summary(d)
sum(abs(d) > 1, na.rm = TRUE)   # count exceeding your 1 keV tolerance

      Min.    1st Qu.     Median       Mean    3rd Qu.       Max. 
-1.067e-04 -2.854e-05 -3.127e-06 -3.694e-06  2.000e-05  6.476e-05 

[1] 0

In [ ]:
subset(df, Z == 26 & A == 56)$BA   # ~8790.34
subset(df, Z == 28 & A == 62)$BA   # ~8794.53

[1] 8790.356

[1] 8794.556

### The binding energy curve

In [ ]:
plot(df$A, df$BA, pch = 20, cex = 0.4,
     xlab = "Mass number A",
     ylab = "Binding energy per nucleon (keV)",
     main = "AME2020, 2550 measured nuclides")

peak <- df[which.max(df$BA), ]
points(peak$A, peak$BA, col = "red", pch = 1, cex = 2)
cat("Maximum: Z =", peak$Z, " A =", peak$A, " B/A =", peak$BA, "\n")

head(df[order(-df$BA), c("Z", "N", "A", "BA")], 5)

In [ ]:
key <- paste(df$Z, df$A)
df$Sn  <- df$B - df$B[match(paste(df$Z, df$A - 1), key)]
df$S2n <- df$B - df$B[match(paste(df$Z, df$A - 2), key)]

cat("Sn computable: ", sum(!is.na(df$Sn)),  "of", nrow(df), "\n")
cat("S2n computable:", sum(!is.na(df$S2n)), "of", nrow(df), "\n")
cat("Negative S2n:  ", sum(df$S2n < 0, na.rm = TRUE), "\n")

Sn computable:  2393 of 2550 
S2n computable: 2301 of 2550 
Negative S2n:   9 


### Phase 1, step 6: cross-check against the AME's own separation energies

The file `rct1.mas20.txt` lists the two-neutron separation energy **S2n** that the AME team computed themselves.
If our numbers match theirs, our binding energies, our neighbour lookups and our subtraction are all correct.

Layout of a data row (from the file header, format `a1,i3,1x,a3,i3,1x,6(f12.4,f10.4)`):

| characters | contents |
|---|---|
| 1 | control character |
| 2–4 | A |
| 6–8 | element symbol |
| 9–11 | Z |
| 13–24 | S2n (keV) |
| 25–34 | its uncertainty (keV) |

A `#` in place of the decimal point means an estimate; a `*` means it could not be calculated.

In [ ]:
url1 <- "https://www-nds.iaea.org/amdc/ame2020/rct1.mas20.txt"
if (!file.exists("rct1.mas20.txt")) download.file(url1, "rct1.mas20.txt")
r1 <- readLines("rct1.mas20.txt")
length(r1)

cat(head(r1, 40), sep = "\n")

In [ ]:
# Keep only data rows: A and Z must both end in a digit
rows1 <- r1[grepl("^.[ 0-9]{2}[0-9] [A-Za-z ]{3}[ 0-9]{2}[0-9] ", r1)]
cat("Data rows in rct1:", length(rows1), "\n")   # should equal the 3558 in the mass table

rct <- data.frame(
  A        = as.integer(substr(rows1,  2,  4)),
  Z        = as.integer(substr(rows1,  9, 11)),
  S2n_text = substr(rows1, 13, 24)
)

# Classify each S2n entry: measured, estimated (#), or not calculable (*)
rct$kind <- ifelse(grepl("*", rct$S2n_text, fixed = TRUE), "not calculable",
            ifelse(grepl("#", rct$S2n_text, fixed = TRUE), "estimated", "measured"))
table(rct$kind)

In [ ]:
# Line up the AME's S2n with ours, nucleus by nucleus
rct$S2n_AME <- ifelse(rct$kind == "measured", suppressWarnings(as.numeric(rct$S2n_text)), NA)
df$S2n_AME  <- rct$S2n_AME[match(paste(df$Z, df$A), paste(rct$Z, rct$A))]

# Does every nucleus have an S2n in exactly the same cases as the AME?
table(ours = !is.na(df$S2n), AME = !is.na(df$S2n_AME))

# How far apart are the numbers where both exist?
diff_S2n <- df$S2n - df$S2n_AME
cat("Compared:", sum(!is.na(diff_S2n)), "nuclei\n")
cat("Largest difference:", max(abs(diff_S2n), na.rm = TRUE), "keV\n")

### The nuclei with negative S2n

A negative S2n means the nucleus **cannot hold its last two neutrons**. These nuclei are past the drip line.
They have still been measured, as short-lived resonances, and every one of them is light.

In [ ]:
unbound <- df[!is.na(df$S2n) & df$S2n < 0, c("Z", "N", "A", "S2n")]
unbound[order(unbound$Z), ]

## Phase 2: fit the five-term formula

$$B(A,Z) = a_V A - a_S A^{2/3} - a_C \frac{Z(Z-1)}{A^{1/3}} - a_A \frac{(A-2Z)^2}{A} + \delta$$

Every constant multiplies something we can compute from A and Z alone, so the fit is a straight
linear least-squares problem. We build the five columns, then solve.

**Choices made here (see `docs/DECISIONS.md`):**
- The main fit uses nuclei with **A ≥ 20**. The liquid-drop picture barely applies to smaller nuclei.
- Everything stays in **keV**. The constants are converted to MeV only when printed.
- There is **no intercept**. The formula has no constant term.

In [ ]:
# The five design columns. Row = nucleus, column = what each constant multiplies.
design <- function(Z, A) {
  parity <- ifelse(A %% 2 == 1, 0,            # odd A: no pairing term
            ifelse(Z %% 2 == 0, +1, -1))      # even-even: +1, odd-odd: -1
  cbind(aV = A,
        aS = -A^(2/3),
        aC = -Z * (Z - 1) / A^(1/3),          # Z(Z-1), not Z^2: a proton does not repel itself
        aA = -(A - 2 * Z)^2 / A,
        aP = parity / sqrt(A))
}

# Quick check on iron-56 (Z = 26, even-even)
design(26, 56)

In [ ]:
# Weighted least squares, written out as matrix steps.
#   w = one weight per nucleus (all 1 for uniform weighting)
# Returns the constants, the raw (X'WX)^-1, and chi-squared per degree of freedom.
fit_semf <- function(d, w) {
  X   <- design(d$Z, d$A)
  y   <- d$B
  XtW <- t(X * w)                        # X' W   (W is diagonal, so multiply each row by its weight)
  C0  <- solve(XtW %*% X)                # (X' W X)^-1
  p   <- drop(C0 %*% XtW %*% y)          # the fitted constants, keV
  r   <- y - drop(X %*% p)               # residuals: measured minus predicted
  dof <- nrow(X) - ncol(X)
  list(p = p, C0 = C0, r = r, n = nrow(X),
       chi2dof = sum(w * r^2) / dof,
       rms = sqrt(mean(r^2)))
}

fitset <- subset(df, A >= 20)
cat("Nuclei in the fit:", nrow(fitset), " (left out with A < 20:", sum(df$A < 20), ")\n")

In [ ]:
# Fit 1: uniform weighting
uni <- fit_semf(fitset, rep(1, nrow(fitset)))

# With equal weights, the size of the scatter is not known in advance, so it is estimated
# from the residuals themselves: C = s^2 (X'X)^-1, where s^2 = residual variance.
C_uni <- uni$C0 * uni$chi2dof

cat("Root-mean-square residual:", round(uni$rms), "keV\n\n")
print(round(rbind(value_MeV = uni$p / 1000,
                  std_error_MeV = sqrt(diag(C_uni)) / 1000), 4))

# Same answer from R's built-in regression? (should print a number near zero)
X <- design(fitset$Z, fitset$A)
max(abs(coef(lm(fitset$B ~ 0 + X)) - uni$p))

### The covariance matrix, as correlations

1 means two constants move together completely, and 0 means not at all.

In [ ]:
round(cov2cor(C_uni), 3)

### Compare with published values

Typical published ranges, in MeV (from the project manual):

In [ ]:
published <- data.frame(
  constant = c("aV", "aS", "aC", "aA", "aP"),
  low  = c(15.5, 16.8, 0.70, 23.0, 11),
  high = c(15.8, 18.3, 0.72, 23.7, 12)
)
published$ours    <- round(uni$p / 1000, 3)
published$inside  <- published$ours >= published$low & published$ours <= published$high
published

### Fit 2: inverse-variance weighting with a floor

Each nucleus gets weight $1/\sigma^2$, but no σ may be smaller than a chosen **floor**.
There is no single right floor, so we try four and see how much the answer moves.

Two things to watch:
1. **χ²/dof** (chi-squared per degree of freedom). It would be about 1 if the formula missed only by the measurement errors. It is far bigger, because the formula itself is wrong by a few MeV.
2. **Raw vs scaled standard errors.** The raw $(X^TWX)^{-1}$ trusts the measurement errors. The scaled version multiplies by χ²/dof, which accounts for how badly the formula really fits.

In [ ]:
floors <- c(1, 10, 100, 1000)   # keV
sens <- do.call(rbind, lapply(floors, function(fl) {
  s <- pmax(fitset$sigma, fl)
  f <- fit_semf(fitset, 1 / s^2)
  data.frame(weighting = "floored", floor_keV = fl,
             chi2_per_dof = f$chi2dof, rms_keV = f$rms,
             aV = f$p[1] / 1000, aS = f$p[2] / 1000, aC = f$p[3] / 1000,
             aA = f$p[4] / 1000, aP = f$p[5] / 1000,
             raw_se_aS    = sqrt(f$C0[2, 2]) / 1000,
             scaled_se_aS = sqrt(f$C0[2, 2] * f$chi2dof) / 1000)
}))
uniform_row <- data.frame(weighting = "uniform", floor_keV = NA,
                          chi2_per_dof = NA, rms_keV = uni$rms,
                          aV = uni$p[1] / 1000, aS = uni$p[2] / 1000, aC = uni$p[3] / 1000,
                          aA = uni$p[4] / 1000, aP = uni$p[5] / 1000,
                          raw_se_aS = NA, scaled_se_aS = sqrt(C_uni[2, 2]) / 1000)
sens <- rbind(uniform_row, sens)
rownames(sens) <- NULL
format(sens, digits = 4)

In [ ]:
# How far does the choice of weighting move each constant, compared with its own error bar?
spread <- apply(sens[, c("aV", "aS", "aC", "aA", "aP")], 2, function(x) max(x) - min(x))
se     <- sqrt(diag(C_uni)) / 1000
round(rbind(spread_MeV = spread, std_error_MeV = se, spread_in_std_errors = spread / se), 3)

### Fit 3: does the A ≥ 20 cutoff matter?

In [ ]:
allset  <- subset(df, Z >= 1 & A - Z >= 1)    # every nucleus except the free neutron and hydrogen-1
all_uni <- fit_semf(allset, rep(1, nrow(allset)))
print(round(rbind("A >= 20"    = uni$p / 1000,
                  "all nuclei" = all_uni$p / 1000,
                  "shift in std errors" = (all_uni$p - uni$p) / sqrt(diag(C_uni))), 3))
cat("Nuclei:", uni$n, "vs", all_uni$n, "\n")
cat("rms: A >= 20:", round(uni$rms), "keV;  all nuclei:", round(all_uni$rms), "keV\n")

### A first look at what the formula misses

Phase 3 studies these residuals properly. This plot is a preview: if the formula captured everything,
the points would be a flat, patternless band around zero. The red dotted lines mark the magic numbers 28, 50, 82 and 126.

In [ ]:
plot(fitset$N, uni$r / 1000, pch = 20, cex = 0.4,
     xlab = "Neutron number N", ylab = "Measured minus predicted B (MeV)",
     main = "Five-term formula residuals (uniform weighting, A >= 20)")
abline(h = 0, col = "grey")
abline(v = c(28, 50, 82, 126), col = "red", lty = 3)